In [ ]:
import time as _rsna_time
RSNA_T0 = _rsna_time.time()  # session start, for the v2 leg deadline


# RSNA knee v2 (ours alone)
Canonical laterality-normalised multi-series 2.5D models trained on gold-calibrated report labels.


In [ ]:
# === v2 leg (our model): canonical laterality-normalised multi-series 2.5D models, rank-blended into the baseline ===
import os, sys, glob, json, subprocess, time, shutil
import numpy as np, pandas as pd
V2_WEIGHT = 1.0          # pre-fixed blend weight for our models (rank space)
V2_ONLY = True              # True -> submit our models alone (no baseline)
_t0 = time.time()
_pkg = [p for pat in ('/kaggle/input/*/kaggle_infer.py', '/kaggle/input/*/*/kaggle_infer.py', '/kaggle/input/datasets/*/*/kaggle_infer.py', '/kaggle/input/datasets/*/*/*/kaggle_infer.py')
        for p in glob.glob(pat) if 'rsna-v2' in p]
_comp = [p for p in [os.environ.get('RSNA_COMP_ROOT', ''), '/kaggle/input/competitions/rsna-knee-abnormality-detection', '/kaggle/input/rsna-knee-abnormality-detection'] if os.path.exists(os.path.join(p, 'test.csv'))][0]
_base_path = '/kaggle/working/submission.csv'
TARGETS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
V2_TAG = 'r20261017-113804-13466'              # release tag of the model package this notebook was built for
V2_MARGIN = 1800                # stop scoring new studies this long before the 9 h limit (unscored -> baseline rank)


def _session_start():
    """Earliest plausible start of this notebook session (the 9 h limit runs from there); errs early, never late."""
    now, cands = time.time(), [_t0]
    if globals().get('RSNA_T0'):
        cands.append(float(globals()['RSNA_T0']))
    try:
        import psutil
        for pid in (os.getpid(), 1):
            try:
                cands.append(psutil.Process(pid).create_time())
            except Exception:
                pass
    except Exception:
        pass
    return min(c for c in cands if 0 <= now - c < 9 * 3600)
if not _pkg:
    raise RuntimeError('v2 model package not mounted')
_tag_file = os.path.join(os.path.dirname(_pkg[0]), 'release_tag.txt')
_mounted = open(_tag_file).read().strip() if os.path.exists(_tag_file) else 'missing'
if _mounted != V2_TAG:
    raise RuntimeError(f'stale v2 model package mounted: tag {_mounted!r} != expected {V2_TAG!r}')
print('v2 package tag OK', V2_TAG, flush=True)
try:
    import gc, torch
    gc.collect(); torch.cuda.empty_cache()
except Exception:
    pass
try:
    pkg = os.path.dirname(_pkg[0])
    models = sorted(glob.glob(os.path.join(pkg, "*.pt")) + glob.glob(os.path.join(pkg, "*", "*.pt")))
    print('v2 package', pkg, len(models), 'models', flush=True)
    _deadline_at = _session_start() + 9 * 3600 - V2_MARGIN
    print(f'v2 deadline in {(_deadline_at - time.time()) / 3600:.2f} h', flush=True)
    cmd = [sys.executable, os.path.join(pkg, 'kaggle_infer.py'), '--comp', _comp, '--models', *models,
           '--out', '/kaggle/working/v2_preds.csv', '--workers', '4', '--fp16', '--pure-fp16',
           '--deadline-at', str(_deadline_at)]
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=max(120, _deadline_at - time.time() + 900))
    print(r.stdout[-3000:], r.stderr[-3000:], flush=True)
    ours = pd.read_csv('/kaggle/working/v2_preds.csv', dtype={'StudyInstanceUID': str})
    test = pd.read_csv(os.path.join(_comp, 'test.csv'), dtype={'StudyInstanceUID': str})
    ours = test[['StudyInstanceUID']].merge(ours, on='StudyInstanceUID', how='left')
    if V2_ONLY or not os.path.exists(_base_path):
        out = ours.copy()
        out[TARGETS] = out[TARGETS].fillna(0.5)
    else:
        base = pd.read_csv(_base_path, dtype={'StudyInstanceUID': str})
        base = test[['StudyInstanceUID']].merge(base, on='StudyInstanceUID', how='left')
        out = base.copy()
        for t in TARGETS:
            b = base[t].rank(pct=True).values
            o = ours[t].rank(pct=True).values
            ok = ~np.isnan(o)
            v = b.copy()
            v[ok] = (1 - V2_WEIGHT) * b[ok] + V2_WEIGHT * o[ok]
            out[t] = v
    assert list(out.columns) == ['StudyInstanceUID'] + TARGETS and len(out) == len(test)
    assert np.isfinite(out[TARGETS].values).all()
    out.to_csv(_base_path, index=False)
    print(f'v2 leg done in {time.time()-_t0:.0f}s, weight {V2_WEIGHT}, only={V2_ONLY}', flush=True)
except Exception as exc:
    import traceback; traceback.print_exc()
    print('v2 leg FAILED; keeping baseline submission', exc, flush=True)


## Third arm: S6 all-data ConvNeXt (nartaa; Heliosli's S6 architecture), OAI component removed
Weights `nartaa/rsna-knee-s6fall-weights` (CC BY-NC 4.0; README: no OAI data; ImageNet-22k init) + Heliosli's public anatomy/code dataset. nartaa's OAI-trained CoAtNet fallback is removed and not attached. Credit: nartaa, Heliosli.


In [ ]:
import base64, os, subprocess, sys, time
import numpy as np, pandas as pd
open('/tmp/s6_stage.py', 'wb').write(base64.b64decode('...'))
W_S6 = {"MCL":0.75,"Lateral Meniscus":0.75,"PF OA":0.75,"Baker's":0.75,"Lateral OA":0.75}  # number, or per-finding dict {finding: S6 weight} (others 0.5)
_left = _session_start() + 9 * 3600 - time.time() - 35 * 60
print(f'S6 stage: {_left / 60:.0f} min left before the margin', flush=True)
_ok = False
if _left > 75 * 60:
    try:
        _env = dict(os.environ, S6_GUARD_S=str(int(_left)), S6_COMP_ROOT=os.environ.get('RSNA_COMP_ROOT', ''))
        _rc = subprocess.run([sys.executable, '-u', '/tmp/s6_stage.py'], env=_env, timeout=_left + 300).returncode
        _ok = _rc == 0 and os.path.isfile('/kaggle/working/s6_sub.csv')
        print('S6 rc', _rc, 'ok', _ok, flush=True)
    except Exception as e:
        print('S6 FAILED', repr(e)[:500], flush=True)
W3 = None  # optional per-finding (R7, S6, ours) rank weights, fixed before any run
try:
    if _ok and W3:
        cur = pd.read_csv('/kaggle/working/submission.csv', dtype={'StudyInstanceUID': str})
        ids = cur[['StudyInstanceUID']]
        rd = lambda f: ids.merge(pd.read_csv(f, dtype={'StudyInstanceUID': str}), on='StudyInstanceUID', how='left')
        a, b, c = rd('/kaggle/working/ours_sub.csv'), rd('/kaggle/working/s6_sub.csv'), rd('/kaggle/working/v2_preds.csv')
        out = cur.copy()
        for t in TARGETS:
            w = W3.get(t, (1/3, 1/3, 1/3)); num = np.zeros(len(ids)); den = np.zeros(len(ids))
            for wk, arm in zip(w, (a, b, c)):
                r = arm[t].rank(pct=True).values; ok = ~np.isnan(r)
                num[ok] += wk * r[ok]; den[ok] += wk
            v = cur[t].rank(pct=True).values.copy(); m_ = den > 0
            v[m_] = num[m_] / den[m_]
            out[t] = v
        out.to_csv('/kaggle/working/submission.csv', index=False)
        print('S6 per-finding W3 blend written', flush=True)
    elif _ok:
        cur = pd.read_csv('/kaggle/working/submission.csv', dtype={'StudyInstanceUID': str})
        s6 = pd.read_csv('/kaggle/working/s6_sub.csv', dtype={'StudyInstanceUID': str})
        s6 = cur[['StudyInstanceUID']].merge(s6, on='StudyInstanceUID', how='left')
        out = cur.copy(); n_ok = 0
        for t in TARGETS:
            o = s6[t].values.astype(float); ok = ~np.isnan(o); n_ok = int(ok.sum())
            if ok.sum() < 2: continue
            c = cur[t].rank(pct=True).values; r = pd.Series(o).rank(pct=True).values
            w6 = W_S6.get(t, 0.5) if isinstance(W_S6, dict) else W_S6
            v = c.copy(); v[ok] = (1 - w6) * c[ok] + w6 * r[ok]
            out[t] = v
        out.to_csv('/kaggle/working/submission.csv', index=False)
        print(f'S6 blended at W_S6={W_S6} on {n_ok} / {len(cur)} studies', flush=True)
    else:
        print('S6 not used: submission = R7 + v2 blend', flush=True)
except Exception as e:
    print('S6 blend FAILED, keeping R7 + v2', repr(e)[:500], flush=True)
